<a href="https://colab.research.google.com/github/Jahnavi-srivalli/BTECH/blob/main/Copy_of_Untitled5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from nltk.corpus import stopwords
import string
from nltk.stem import PorterStemmer
from geopy.geocoders import Nominatim
import nltk

In [ ]:
# Download NLTK stopwords if not already downloaded
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# Load dataset
df = pd.read_csv("/content/MOCK_DATA1.csv")

In [ ]:
# Function to preprocess text data
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

In [ ]:
#preprocessing the text
def preprocess_text(text):
    text = text.lower()  # Lowercase text
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    tokens = text.split()  # Tokenize text
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]  # Stemming and remove stopwords
    return ' '.join(tokens)

In [ ]:
# Function to preprocess location data
def preprocess_location(location_str):
    geolocator = Nominatim(user_agent="traffic_flow_prediction")
    location = geolocator.geocode(location_str)
    if location is None:
        raise ValueError(f"Could not geocode location: {location_str}")
    return location.latitude, location.longitude

In [ ]:
# Apply preprocessing to text data
df['text'] = df['text'].apply(preprocess_text)

In [ ]:
# Split dataset into training and testing sets
X_geo = df[['latitude', 'longitude']]
X_text = df['text']
y = df['trafficflow']
X_train_geo, X_test_geo, X_train_text, X_test_text, y_train, y_test = train_test_split(X_geo, X_text, y, test_size=0.2, random_state=42)

In [ ]:
# Define parameter grid for grid search
param_grid = {
    'tfidfvectorizer__max_features': [1000, 2000, 5000],
    'tfidfvectorizer__ngram_range': [(1, 1), (1, 2)],
    'randomforestclassifier__n_estimators': [50, 100, 200]
}

In [ ]:
# Create pipelines for both geographical and textual data with improved vectorization
pipeline_geo = make_pipeline(RandomForestClassifier(random_state=42))

pipeline_text = make_pipeline(
    TfidfVectorizer(),
    RandomForestClassifier(random_state=42)
)

In [ ]:
# Perform grid search to find the best parameters for textual model
grid_search = GridSearchCV(pipeline_text, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_text, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tfidfvectorizer', TfidfVectorizer()),
                                       ('randomforestclassifier',
                                        RandomForestClassifier(random_state=42))]),
             param_grid={'randomforestclassifier__n_estimators': [50, 100, 200],
                         'tfidfvectorizer__max_features': [1000, 2000, 5000],
                         'tfidfvectorizer__ngram_range': [(1, 1), (1, 2)]},
             scoring='accuracy')

In [ ]:
# Get the best model from grid search
best_text_model = grid_search.best_estimator_

In [ ]:
# Evaluate geographical model using cross-validation
geo_scores = cross_val_score(pipeline_geo, X_train_geo, y_train, cv=5, scoring='accuracy')
print(f"Geographical Model Cross-Validation Scores: {geo_scores}")
print(f"Geographical Model Mean CV Score: {geo_scores.mean()}")

Geographical Model Cross-Validation Scores: [0.4625  0.45625 0.53125 0.45    0.48125]
Geographical Model Mean CV Score: 0.47625


In [ ]:
# Evaluate textual model using cross-validation
text_scores = cross_val_score(best_text_model, X_train_text, y_train, cv=5, scoring='accuracy')
print(f"Textual Model Cross-Validation Scores: {text_scores}")
print(f"Textual Model Mean CV Score: {text_scores.mean()}")

Textual Model Cross-Validation Scores: [0.975   0.95625 0.96875 0.9625  0.9375 ]
Textual Model Mean CV Score: 0.96


In [ ]:
# Train models on the entire training data
pipeline_geo.fit(X_train_geo, y_train)
best_text_model.fit(X_train_text, y_train)

Pipeline(steps=[('tfidfvectorizer', TfidfVectorizer(max_features=1000)),
                ('randomforestclassifier',
                 RandomForestClassifier(n_estimators=50, random_state=42))])

In [ ]:
# Function to predict traffic flow based on user input location and text
def predict_traffic_flow(location_str, text_str):
    try:
        # Preprocess location and text
        lat, lon = preprocess_location(location_str)
        text_processed = preprocess_text(text_str)

        # Predict using both models
        prediction_geo = pipeline_geo.predict([[lat, lon]])
        prediction_text = best_text_model.predict([text_processed])

        # Print intermediate predictions
        print(f"Intermediate Predictions:")
        print(f"Geographical Model Prediction: {prediction_geo}")
        print(f"Textual Model Prediction: {prediction_text}")

        # Combine predictions (simple averaging for demonstration)
        combined_prediction = (prediction_geo + prediction_text) / 2

        # Final prediction based on threshold (0.5)
        final_prediction = 0 if combined_prediction < 0.5 else 1

        return final_prediction

    except ValueError as e:
        # Handle cases where location could not be geocoded
        print(f"location is not correct")
        return None
    except Exception as e:
        # Handle any other exceptions that may occur
        print(f"other exceptions")
        return None

In [ ]:
# User input for location and text
user_location = input("Enter city: ")
user_text = input("Enter text: ")

# Predicted traffic flow
predicted_traffic = predict_traffic_flow(user_location, user_text)

if predicted_traffic is not None:
    traffic_status = "Smooth Traffic" if predicted_traffic == 0 else "Heavy Traffic"
    print(f"\nFinal Predicted traffic flow for {user_location}: {traffic_status}")
else:
    print(f"\nCould not predict traffic flow for {user_location}. Please check the location.")

Enter city: tanuku
Enter text: busy hours
Intermediate Predictions:
Geographical Model Prediction: [1]
Textual Model Prediction: [1]

Final Predicted traffic flow for tanuku: Heavy Traffic


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import nltk
from nltk.corpus import stopwords
import string
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

nltk.download('stopwords')

# Load dataset
df = pd.read_csv("/content/MOCK_DATA (10).csv")
df1 = df.drop_duplicates()

# Function to preprocess text data
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    text = text.lower()  # Lowercase text
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    tokens = text.split()  # Tokenize text
    tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
    return ' '.join(tokens)

# Preprocess location function with error handling
def preprocess_location(location_str):
    geolocator = Nominatim(user_agent="traffic_prediction_app")
    try:
        location = geolocator.geocode(location_str)
        if location:
            return location.latitude, location.longitude
    except GeocoderTimedOut:
        return None, None
    return None, None

# Apply preprocessing to text data
df1['text'] = df1['text'].apply(preprocess_text)

# Split dataset into training and testing sets
X = df1[['latitude', 'longitude', 'text']]
y = df1['trafficflow']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Column transformer for combining geographical and text features
geo_preprocessor = ColumnTransformer(
    transformers=[
        ('geo', StandardScaler(), ['latitude', 'longitude'])
    ])

text_preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer()), 'text'
    ])

# Create separate pipelines for geographical and text features
geo_pipeline = make_pipeline(
    geo_preprocessor,
    RandomForestClassifier(random_state=42)
)

text_pipeline = make_pipeline(
    TfidfVectorizer(),
    RandomForestClassifier(random_state=42)
)

# Hyperparameter tuning using GridSearchCV
param_grid = {
    'randomforestclassifier__n_estimators': [100, 200],
    'randomforestclassifier__max_depth': [10, 20]
}

geo_grid_search = GridSearchCV(geo_pipeline, param_grid, cv=5)
geo_grid_search.fit(X_train[['latitude', 'longitude']], y_train)

text_grid_search = GridSearchCV(text_pipeline, param_grid, cv=5)
text_grid_search.fit(X_train['text'], y_train)

# Evaluate the models
y_geo_pred = geo_grid_search.predict(X_test[['latitude', 'longitude']])
y_text_pred = text_grid_search.predict(X_test['text'])

print("Geographical Model Accuracy:", accuracy_score(y_test, y_geo_pred))
print("Geographical Model Classification Report:\n", classification_report(y_test, y_geo_pred))

print("Textual Model Accuracy:", accuracy_score(y_test, y_text_pred))
print("Textual Model Classification Report:\n", classification_report(y_test, y_text_pred))

# Function to predict traffic flow based on user input location and text
def predict_traffic_flow(location_str, text_str):
    lat, lon = preprocess_location(location_str)
    if lat is None or lon is None:
        return "Invalid location"
    text_processed = preprocess_text(text_str)

    # Create DataFrames for the inputs
    geo_input_df = pd.DataFrame([[lat, lon]], columns=['latitude', 'longitude'])
    text_input_df = pd.DataFrame([text_processed], columns=['text'])

    # Use the trained pipelines to predict
    geo_prediction = geo_grid_search.predict(geo_input_df)
    text_prediction = text_grid_search.predict(text_input_df['text'])

    geo_result = "Smooth Traffic" if geo_prediction[0] == 0 else "Heavy Traffic"
    text_result = "Smooth Traffic" if text_prediction[0] == 0 else "Heavy Traffic"

    # Majority voting for final prediction
    predictions = [geo_prediction[0], text_prediction[0]]
    final_prediction = max(set(predictions), key=predictions.count)
    final_result = "Smooth Traffic" if final_prediction == 0 else "Heavy Traffic"

    return geo_result, text_result, final_result

# User inputs
user_location = input("Enter city: ")
user_text = input("Enter traffic description: ")

latitude, longitude = preprocess_location(user_location)
if latitude is not None:
    print(f"Latitude: {latitude}")
else:
    print("Could not determine latitude.")
if longitude is not None:
    print(f"Longitude: {longitude}")
else:
    print("Could not determine longitude.")

geo_predicted_traffic, text_predicted_traffic, final_predicted_traffic = predict_traffic_flow(user_location, user_text)
print(f"Geographical Model Prediction for {user_location}: {geo_predicted_traffic}")
print(f"Textual Model Prediction: {text_predicted_traffic}")
print(f"Final Prediction: {final_predicted_traffic}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Geographical Model Accuracy: 0.525
Geographical Model Classification Report:
               precision    recall  f1-score   support

           0       0.54      0.54      0.54       103
           1       0.51      0.51      0.51        97

    accuracy                           0.53       200
   macro avg       0.52      0.52      0.52       200
weighted avg       0.52      0.53      0.52       200

Textual Model Accuracy: 1.0
Textual Model Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       103
           1       1.00      1.00      1.00        97

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

Enter city: tanuku
Enter traffic description: low
Latitude: 16.75414
Longitude: 81.685839
Geographical Model Prediction for tanuku: Smooth Traffic
Textual Model Prediction: Smooth Traffic
Final Predicti

In [ ]:
'''predicted_traffic = predict_traffic_flow(user_location, user_text)
traffic_status = "Smooth Traffic" if predicted_traffic == 0 else "Heavy Traffic"
print(f"\nFinal Predicted traffic flow for {user_location}: {traffic_status}")'''

# User input for location and text
user_location = input("Enter city: ")
user_text = input("Enter text: ")

# Predicted traffic flow
predicted_traffic = predict_traffic_flow(user_location, user_text)
if predicted_traffic is not None:
    traffic_status = "Smooth Traffic" if predicted_traffic == 0 else "Heavy Traffic"
    print(f"\nFinal Predicted traffic flow for {user_location}: {traffic_status}")
else:
    print(f"\nCould not predict traffic flow for {user_location}. Please check the location.")